In [0]:
from pyspark.sql import functions as F

silver_orders_new = spark.read.table("silver_orders_new")
display(silver_orders_new)
gold_orders_new = (
    silver_orders_new
    .groupBy('customer_name')
    .agg(
        F.count('order_id').alias('total_orders'), 
        F.round(F.sum('amount'), 2).alias('total_spent'), 
        F.round(F.avg('amount'),2).alias('avg_order_value')
    )
    .withColumn('customer_status',F.when(F.col('total_spent') >= 1000, 'Platinum').otherwise('Standard'))
    .orderBy(F.col('total_spent').desc())
)
display(gold_orders_new)

(
    gold_orders_new
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('gold_orders_new')
)


customer_name,product,order_id,customer_id,amount,order_date,ingestion_timestamp
John Doe,LAPTOP,101,C101,1500.5,2026-10-01,2026-10-05T18:26:35.304Z
Jane Smith,PHONE,102,C102,750.0,2026-10-01,2026-10-05T18:26:35.304Z
John Doe,HEADPHONES,105,C101,50.0,2026-10-03,2026-10-05T18:26:35.304Z
John Doe,LAPTOP,106,C105,1200.0,2026-10-03,2026-10-05T18:26:35.304Z
Jane Smith,MONITOR,107,C102,300.0,2026-10-04,2026-10-05T18:26:35.304Z
Alice Johnson,KEYBOARD,108,C106,85.5,2026-10-04,2026-10-05T18:26:35.304Z
Bob Brown,MOUSE,109,C107,25.0,2026-10-05,2026-10-05T18:26:35.304Z
Charlie Davis,null,110,C108,150.0,2026-10-05,2026-10-05T18:26:35.304Z
Alice Johnson,DESK LAMP,111,C106,45.0,2026-10-05,2026-10-05T18:26:35.304Z
David Miller,GAMING CHAIR,112,C109,350.0,2026-10-06,2026-10-05T18:26:35.304Z


customer_name,total_orders,total_spent,avg_order_value,customer_status
John Doe,4,2765.5,691.38,Platinum
Jane Smith,3,1062.5,354.17,Platinum
David Miller,2,650.0,325.0,Standard
Henry Taylor,1,210.0,210.0,Standard
Charlie Davis,1,150.0,150.0,Standard
Alice Johnson,2,130.5,65.25,Standard
Bob Brown,2,120.0,60.0,Standard
Emma Wilson,1,120.0,120.0,Standard
Grace Lee,1,110.0,110.0,Standard
